# Oversampled 4th-power versus band-edge FLL

This notebook is the slower companion to `notes/oversampled-fourth-power-vs-band-edge-fll.md`.

The narrow question is the useful one: **before late-stage Costas tracking, when is oversampled 4th-power the cleaner object, and when is a band-edge discriminator the cleaner object?**


## The two clues are different

The 4th-power branch still exploits PSK rotational symmetry.
The band-edge branch exploits excess-bandwidth asymmetry in the pulse-shaped waveform.

That sounds simple, but it is the whole decision boundary. The front ends are not interchangeable "coarse loops."


## The alias limit for the oversampled 4th-power branch

For QPSK 4th-power estimation the honest per-sample phase range is still `|\omega| < \pi / 4`.
If the waveform is still at `L` samples per symbol, then

```text
|Δf| < L R_s / 8
```

So at `4 sps` the alias boundary lands at `|Δf| / R_s < 0.5`.

That boundary depends on sample rate. The band-edge branch depends on roll-off.


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

rows = []
with (Path('..') / 'assets' / '2026-05-19-oversampled-fourth-power-vs-band-edge-fll.csv').open() as handle:
    for row in csv.DictReader(handle):
        rows.append({key: float(value) for key, value in row.items()})

len(rows), rows[0].keys()


## Read one fixed offset

A good bounded checkpoint is `Δf / R_s = 0.10`.
The oversampled 4th-power estimate should stay close to `0.10` for every roll-off.
The band-edge imbalance should grow as `α` grows.


In [ ]:
target = [row for row in rows if abs(row['normalized_cfo'] - 0.10) < 1e-9]
for row in sorted(target, key=lambda row: row['rolloff']):
    print({
        'alpha': row['rolloff'],
        'fourth_power_estimate': round(row['fourth_power_estimate'], 4),
        'fourth_power_abs_error': round(row['fourth_power_absolute_error'], 4),
        'band_edge_imbalance': round(row['band_edge_imbalance'], 4),
    })


That is the split in one table:

- the symmetry-based estimate barely moves
- the waveform-domain clue gets much stronger when roll-off leaves more edge energy


## Check the oversampled 4th-power lane near the 4 sps boundary

Inside `|Δf| / R_s <= 0.45`, the estimate should stay close to the identity line.
At the actual `0.50` boundary the branch is no longer owed perfect honesty.


In [ ]:
inside = [row for row in rows if abs(row['normalized_cfo']) <= 0.45 + 1e-9]
worst = max(inside, key=lambda row: row['fourth_power_absolute_error'])
worst


That worst case is still small. So this pass does **not** say the oversampled symmetry-based branch becomes fragile as soon as pulse shaping changes.

It says something narrower:

1. the oversampled 4th-power branch still behaves like the same estimator family
2. the band-edge branch becomes attractive for a different reason entirely


## Caveat

The band-edge value in this notebook is a bounded imbalance metric, not a full closed-loop FLL implementation.
That is intentional. The first honest question is whether the clue exists and whether its strength tracks excess bandwidth.

Adjacent-channel leakage, loop dynamics, and implementation details belong to a later, heavier pass.


## Problems worth keeping

1. Add matched filtering and timing recovery to the same waveform experiment. Does the comparison stay as clean?
2. Keep the same roll-off sweep but add one pilot or preamble branch as a decision-box comparison, not a sprawling third simulation packet.
3. Regenerate one older SDR visual with the same rebuildable figure path so the whole synchronization cluster stays coherent.
